# 人类环节：预执行门控、风险分层与审计日志

本课的自述文件介绍了人类环节（Human-in-the-Loop），通过一个简短的代码片段让用户在代理已经给出响应后选择 `APPROVE` 或 `REJECT`。这种模式是一个很好的起点，但在生产环境中，HITL 实现通常需要三个额外部分：

1. 一个 <strong>预执行门控</strong>，在代理执行风险步骤 <strong>之前</strong> 运行，以确保成本、不可逆性和延迟可控。
2. <strong>风险分层</strong>，使低风险操作自动执行，中风险操作批量批准，只有高风险操作才需要人工阻断。
3. 一个 <strong>审计日志加修订循环</strong>，记录每个门控决策为 JSONL 格式，拒绝时用结构化的理由重新提示代理，而不是简单打印 `Revising...`。

本笔记本基于与 `06-system-message-framework.ipynb` 相同的原语逐步构建这些功能。它可以在 `DEMO_MODE = True` 下端到端运行（无需交互输入），也可以在 `DEMO_MODE = False` 时通过实际的 `input()` 提示运行。注意：在 DEMO_MODE 中，第三项目标的重试是脚本化的，使循环机制端到端可见。真实的基于修订的重新分类需要 `DEMO_MODE = False` 并由操作员介入。

**不在本课范围之内（由其他课程涵盖）：** 认证和访问控制（第06课自述中的威胁2），工具调用中间件（第14课MAF深入讲解），多代理辩论模式。

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: 预执行门控

README 中的 HITL 代码片段先调用代理，然后请求用户批准输出。这是一个<strong>事后执行</strong>流程。代理已经执行，因此 LLM 调用费用已经产生，且任何副作用（发送的邮件、写入的数据库行、发布的评论）也已经发生。

<strong>预执行</strong>流程会在代理运行风险步骤之前插入门控。代理提出操作，门控决定是否执行，只有获得批准时副作用才会发生。

| 方面 | 事后批准（README 代码片段） | 预执行门控（本笔记本） |
|---|---|---|
| 何时运行批准？ | 代理产生输出之后 | 任何副作用执行之前 |
| 拒绝时 LLM 费用 | 已经支付 | 只支付提议费用，不支付操作费用 |
| 不可逆副作用 | 可能（操作已发生） | 被阻止 |
| 审计清晰度 | 批准为打印语句 | 批准为带时间戳、操作、原因的 JSON 记录 |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: 风险分级

并非每个操作都需要人工批准。对公共 API 进行只读查询的风险与发送客户电子邮件的风险不同。将两者视为相同会浪费操作员的注意力并减慢代理速度。

一个简单的三层模型：

| 级别 | 示例 | 批准流程 |
|---|---|---|
| `低`（只读） | 搜索知识库，查询航班选项，获取公共网页 | 自动执行，记录审计 |
| `中`（廉价变更） | 缓存结果，增加计数器，安排提醒 | 自动执行，但每天集中复审 |
| `高`（对外或不可逆） | 发送电子邮件，扣费，发布到公共频道 | 阻止，需人工批准 |

这是一种分级方式。生产系统通常使用更细粒度的等级（例如，AWS IAM 权限级别、基于角色的访问等级）。以下三层版本是结合只读和有副作用操作的代理中最小实用版本。

下面的分类器使用关键词启发式，以确保演示保持确定性和低成本。在生产系统中，您会用学习型分类器或策略引擎替代它。

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## 模式 3：审计日志与修订循环

`print("Response approved.")` 并不是审计日志。为了建立信任，每个关卡决策都应作为结构化事件记录下来，以便以后查询、重放或附加到事件审查中。

两个部分：

1. **追加式 JSONL。** 每个决策一行，包含时间戳、动作、等级、决策、理由。方便 grep，方便以后发送到真正的日志存储。
2. **拒绝时的修订循环。** 当关卡返回 `deny` 时，代理会带着拒绝理由重新提示自己，这样下一次提案就能避免出现问题。

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## 额外资源

还有几个公共项目实现了这些 HITL 模式的变体。比较这些方法以找到适合你技术栈的方案：

- **LangChain** 人工参与工具包装器 ([文档](https://python.langchain.com/docs/integrations/tools/human_tools))：即插即用的工具包装器，执行时暂停以等待人工输入。
- **AutoGen** `UserProxyAgent` ([v0.2 文档](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ 对此结构进行了重构)：使用一个专门代表人工的代理角色来进行多代理对话。
- **Semantic Kernel** 函数过滤器 ([文档](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters))：环绕每个函数调用运行的中间件，适用于门控逻辑。

每个项目对这三个子模式的处理方式不同：LangChain 将它们包装为工具，AutoGen 使用代理角色，Semantic Kernel 使用中间件过滤器。在为自己的代理选择设计之前，先完整阅读一两个实现。

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
